# SRAG Mossoro - Analise e Previsao (Notebook Unico)

Este notebook cobre o fluxo fim a fim: carga de dados, analise epidemiologica e previsao de curto prazo.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sqlite3

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error

plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
def find_repo_root(start: Path | None = None) -> Path:
    curr = (start or Path.cwd()).resolve()
    for p in [curr, *curr.parents]:
        if (p / 'pyproject.toml').exists():
            return p
    raise FileNotFoundError('Nao foi possivel localizar a raiz do repositorio (pyproject.toml).')

ROOT = find_repo_root()
DB_PATH = ROOT / 'data' / 'srag_mossoro.db'
CSV_FALLBACK = ROOT / 'data' / 'srag_mossoro_secure.csv'

WEEKS_TO_PREDICT = 4
HOLDOUT_WEEKS = 4

print(f'ROOT: {ROOT}')
print(f'DB_PATH: {DB_PATH}')
print(f'CSV_FALLBACK: {CSV_FALLBACK}')

In [ ]:
import sys

SRC_PATH = ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from srag.data.analytics import (
    compute_age_groups,
    compute_severity_metrics,
    compute_time_series,
    compute_virus_distribution,
)
from srag.models.forecasting import predict_next_weeks
from srag.viz import (
    plot_age_groups,
    plot_history_with_forecast,
    plot_time_series,
    plot_virus_distribution,
)

In [ ]:
def load_cases_dataframe(db_path: Path, csv_fallback: Path) -> pd.DataFrame:
    if db_path.exists():
        with sqlite3.connect(db_path) as conn:
            df = pd.read_sql_query('SELECT * FROM casos_srag', conn)
        date_cols = ['dt_notific', 'dt_sin_pri']
        for c in date_cols:
            if c in df.columns:
                df[c] = pd.to_datetime(df[c], errors='coerce').dt.date
        return df

    if csv_fallback.exists():
        df = pd.read_csv(csv_fallback)
        for c in ['dt_notific', 'dt_sin_pri']:
            if c in df.columns:
                df[c] = pd.to_datetime(df[c], errors='coerce').dt.date
        return df

    raise FileNotFoundError(
        f'Dados nao encontrados. Gere o banco em {db_path} ou um CSV em {csv_fallback}.'
    )

In [ ]:
df = load_cases_dataframe(DB_PATH, CSV_FALLBACK)

print(f'Total de registros: {len(df)}')
print('Colunas:', ', '.join(df.columns))
df.head()

## 1) Qualidade e consistencia dos dados

In [ ]:
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
missing_pct.head(15).to_frame('missing_pct_top15')

In [ ]:
if 'dt_sin_pri' in df.columns:
    invalid_dates = df['dt_sin_pri'].isna().sum()
    print(f'Datas de inicio de sintomas invalidas/ausentes: {invalid_dates}')

if 'idade_anos' in df.columns:
    invalid_age = (pd.to_numeric(df['idade_anos'], errors='coerce') < 0).sum()
    print(f'Idades negativas: {invalid_age}')

## 2) Analise epidemiologica

In [ ]:
metrics = compute_severity_metrics(df.copy())
pd.DataFrame([metrics])

In [ ]:
virus_df = compute_virus_distribution(df.copy())
virus_df = virus_df.sort_values('count', ascending=False)
virus_df

In [ ]:
ax = plot_virus_distribution(virus_df)
if ax is not None:
    plt.show()

In [ ]:
age_df = compute_age_groups(df.copy())
age_df = age_df.sort_values('count', ascending=False)
age_df

In [ ]:
ax = plot_age_groups(age_df)
if ax is not None:
    plt.show()

In [ ]:
ts_df = compute_time_series(df.copy())
ts_df.tail(12)

In [ ]:
ax = plot_time_series(ts_df)
if ax is not None:
    plt.show()

## 3) Previsao de curto prazo

In [ ]:
forecast_result = predict_next_weeks(ts_df.copy(), weeks_to_predict=WEEKS_TO_PREDICT)

history_df = pd.DataFrame(forecast_result.get('history', []))
forecast_df = pd.DataFrame(forecast_result.get('forecast', []))

print('status:', forecast_result.get('status'))
print('model_type:', forecast_result.get('model_type'))
forecast_df

In [ ]:
ax = plot_history_with_forecast(history_df, forecast_df)
if ax is not None:
    plt.show()

## 4) Validacao simples (holdout)

In [ ]:
def evaluate_holdout(ts: pd.DataFrame, holdout_weeks: int = 4) -> pd.DataFrame:
    if len(ts) < max(8, holdout_weeks + 4):
        return pd.DataFrame()

    train = ts.iloc[:-holdout_weeks].copy()
    test = ts.iloc[-holdout_weeks:].copy()

    pred = predict_next_weeks(train, weeks_to_predict=holdout_weeks)
    pred_df = pd.DataFrame(pred.get('forecast', []))
    if pred_df.empty:
        return pd.DataFrame()

    out = test[['epi_week', 'total_cases']].reset_index(drop=True).merge(
        pred_df[['epi_week', 'predicted_cases']], on='epi_week', how='left'
    )
    return out

eval_df = evaluate_holdout(ts_df, holdout_weeks=HOLDOUT_WEEKS)
eval_df

In [ ]:
if not eval_df.empty and eval_df['predicted_cases'].notna().all():
    y_true = eval_df['total_cases'].to_numpy()
    y_pred = eval_df['predicted_cases'].to_numpy()

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    print(f'MAE: {mae:.2f}')
    print(f'RMSE: {rmse:.2f}')
else:
    print('Nao foi possivel calcular metricas de holdout (dados insuficientes ou semanas sem alinhamento).')

## 5) Conclusoes operacionais

Preencha ao final da execucao:

- Tendencia atual (alta, queda, estabilidade): ...
- Faixas etarias prioritarias: ...
- Classificacao viral dominante: ...
- Recomendacoes para vigilancia (curto prazo): ...